In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from scipy.optimize import minimize
from scipy.stats import norm

In [ ]:
features = ["PT1", "PT2", "P1", "P2", "TotalPT", "VertexChisq", "Isolation", "MASS"]
background = pd.read_csv("data/background_combinatorial.txt", sep=r"\s+", header=None, names=features)
background = background.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
bkg_mass = background["MASS"].to_numpy()

with open("bdt_results.json") as fh:
    bdt = json.load(fh)

# prefer the Punzi working point from the extension if present
if "signal_efficiency_punzi" in bdt:
    sig_eff = bdt["signal_efficiency_punzi"]
    bkg_eff = bdt["background_efficiency_punzi"]
    wp = "Punzi"
else:
    sig_eff = bdt["signal_efficiency"]
    bkg_eff = bdt["background_efficiency"]
    wp = "baseline"

print(f"{wp} working point: sig_eff = {sig_eff:.4f}, bkg_eff = {bkg_eff:.4f}")

(a)

In [ ]:
MASS_LO, MASS_HI = 4.0, 6.0


def exp_pdf(m, lam):
    return np.exp(-lam * m) / ((np.exp(-lam * MASS_LO) - np.exp(-lam * MASS_HI)) / lam)


def nll_bkg(params, masses):
    lam = params[0]
    if lam <= 0:
        return 1e10
    return -np.sum(np.log(exp_pdf(masses, lam)))


lam_fit = float(minimize(nll_bkg, x0=[0.5], args=(bkg_mass,), method="Nelder-Mead").x[0])
print(f"lambda = {lam_fit:.4f}")

with open("fit_params.json", "w") as fh:
    json.dump({"lam_bkg": lam_fit}, fh, indent=2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(MASS_LO, MASS_HI, 60)
ax.hist(bkg_mass, bins=bins, density=True, alpha=0.6, label="background")
m_range = np.linspace(MASS_LO, MASS_HI, 300)
ax.plot(m_range, exp_pdf(m_range, lam_fit), color="red", label=f"lambda = {lam_fit:.3f}")
ax.set_xlabel("mass [GeV]"); ax.set_ylabel("density"); ax.legend()
plt.tight_layout()
plt.savefig("plots/background_fit.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
MU_SIG, SIG_SIG = 5.0, 0.03
N_SIG_YEAR = 50 * sig_eff
N_BKG_YEAR = 2000 * bkg_eff
print(f"per year: {N_SIG_YEAR:.1f} signal, {N_BKG_YEAR:.1f} background")


def sample_exp(lam, n, rng):
    u = rng.uniform(0, 1, size=n)
    return -np.log(np.exp(-lam * MASS_LO) - u * (np.exp(-lam * MASS_LO) - np.exp(-lam * MASS_HI))) / lam


def generate_toy(n_sig_mean, n_bkg_mean, lam, rng):
    masses = np.concatenate([
        rng.normal(MU_SIG, SIG_SIG, rng.poisson(n_sig_mean)),
        sample_exp(lam, rng.poisson(n_bkg_mean), rng),
    ])
    return masses[(masses >= MASS_LO) & (masses <= MASS_HI)]


rng = np.random.default_rng(42)
N_TOYS = 1000
toys = [generate_toy(N_SIG_YEAR, N_BKG_YEAR, lam_fit, rng) for _ in range(N_TOYS)]

(b)

In [ ]:
def composite_pdf(m, fs, lam):
    return fs * norm.pdf(m, MU_SIG, SIG_SIG) + (1 - fs) * exp_pdf(m, lam)


def nll_composite(params, masses):
    fs, lam = params
    if fs < 0 or fs > 1 or lam <= 0:
        return 1e10
    v = composite_pdf(masses, fs, lam)
    if np.any(v <= 0):
        return 1e10
    return -np.sum(np.log(v))


def fit_toy(masses):
    r = minimize(nll_composite, x0=[0.02, lam_fit], args=(masses,), method="Nelder-Mead",
                 options={"xatol": 1e-5, "fatol": 1e-5, "maxiter": 5000})
    return float(r.x[0]), float(r.x[1]), float(r.fun)


f_true = N_SIG_YEAR / (N_SIG_YEAR + N_BKG_YEAR)
fit_results = [fit_toy(toy) for toy in toys]
fs_vals = np.array([r[0] for r in fit_results])
nll1_vals = np.array([r[2] for r in fit_results])
print(f"f_true = {f_true:.4f}, <f_s> = {fs_vals.mean():.4f} +/- {fs_vals.std():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(fs_vals, bins=40, alpha=0.7)
ax.axvline(f_true, color="red", linestyle="--", label=f"true {f_true:.4f}")
ax.axvline(fs_vals.mean(), color="black", linestyle="--", label=f"mean {fs_vals.mean():.4f}")
ax.set_xlabel("fitted signal fraction"); ax.set_ylabel("toys"); ax.legend()
plt.tight_layout()
plt.savefig("plots/toy_signal_fractions.png", dpi=150, bbox_inches="tight")
plt.show()

(c)

In [ ]:
def nll_bkg_only(params, masses):
    lam = params[0]
    if lam <= 0:
        return 1e10
    return -np.sum(np.log(exp_pdf(masses, lam)))


def fit_bkg_only(masses):
    r = minimize(nll_bkg_only, x0=[lam_fit], args=(masses,), method="Nelder-Mead")
    return float(r.fun)


nll0_vals = np.array([fit_bkg_only(toy) for toy in toys])
q_vals = np.clip(2 * (nll0_vals - nll1_vals), 0, None)
significance = np.sqrt(q_vals)
print(f"median Z = {np.median(significance):.2f}, frac > 5 sigma = {(significance > 5).mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(significance, bins=40, alpha=0.7)
ax.axvline(5, color="red", linestyle="--", label="5 sigma")
ax.set_xlabel("significance [sigma]"); ax.set_ylabel("toys"); ax.legend()
plt.tight_layout()
plt.savefig("plots/toy_significance.png", dpi=150, bbox_inches="tight")
plt.show()

(d)

In [ ]:
def prob_discovery_at(T, n_toys=500, rng_seed=0):
    rng = np.random.default_rng(rng_seed)
    ns, nb_ = 50 * sig_eff * T, 2000 * bkg_eff * T
    sigs = []
    for _ in range(n_toys):
        toy = generate_toy(ns, nb_, lam_fit, rng)
        _, _, nll1 = fit_toy(toy)
        nll0 = fit_bkg_only(toy)
        sigs.append(np.sqrt(max(0.0, 2 * (nll0 - nll1))))
    return float((np.array(sigs) > 5).mean())


durations = np.round(np.arange(0.1, 1.55, 0.1), 1)
probs = np.array([prob_discovery_at(T, rng_seed=int(T * 100)) for T in durations])
for T, p in zip(durations, probs):
    print(f"T = {T:.1f} yr  P(>5 sigma) = {p:.3f}")

In [ ]:
idx = np.where(probs >= 0.95)[0]
T95 = durations[idx[0]] if len(idx) else None
print("T95 =", T95, "yr")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(durations, probs, marker="o", markersize=4)
ax.axhline(0.95, color="red", linestyle="--", label="95%")
ax.set_xlabel("experiment duration [years]"); ax.set_ylabel("P(Z > 5 sigma)"); ax.legend()
plt.tight_layout()
plt.savefig("plots/discovery_duration.png", dpi=150, bbox_inches="tight")
plt.show()